# Siamese Neural Network for Resume-Job Description Matching

This notebook implements a resume-job description matching system using a Siamese Neural Network with Sentence-BERT. It calculates the similarity between resumes and a given job description, excluding the manual ranking columns.

In [51]:
%pip install sentence-transformers pandas numpy nltk scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [52]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [53]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## 1. Data Loading and Preprocessing

In [54]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

In [55]:
resume_df = pd.read_csv(r'D:\PS-2-V2\PS-2 functions\data\raw\Manual Ranking SNN.csv')
with open(r'D:\PS-2-V2\PS-2 functions\data\job_descriptions\Web-Developer-job-description.txt', 'r') as f:
    job_description = f.read()

In [56]:
resume_df['combined_text'] = resume_df.apply(lambda row: ' '.join(row.drop(['Name', 'Contact Email', 'Phone Number', 'LinkedIn Profile', 'GitHub Profile', 'hyperlinks', 'Domain', 'Perfect', 'Good', 'Average', 'Poor', 'Kush', 'Kapil', 'Amartya']).astype(str)), axis=1)
resume_df['processed_text'] = resume_df['combined_text'].apply(preprocess_text)
job_description_processed = preprocess_text(job_description)

## 2. Siamese Network Matching with SBERT

In [57]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [58]:
resume_embeddings = model.encode(resume_df['processed_text'].tolist(), convert_to_tensor=True)
job_description_embedding = model.encode([job_description_processed], convert_to_tensor=True)

In [59]:
similarities = cosine_similarity(resume_embeddings.cpu().numpy(), job_description_embedding.cpu().numpy())
resume_df['similarity_score'] = similarities

## 3. Results

In [60]:
ranked_resumes = resume_df.sort_values(by='similarity_score', ascending=False)
print('All Resumes Ranked by Similarity Score:')
print(ranked_resumes[['Name', 'similarity_score']])

All Resumes Ranked by Similarity Score:
                           Name  similarity_score
129                 Divij Goyal          0.668200
73               Nishant Sharma          0.653057
80         PATLOLA ANUDEEPREDDY          0.646466
105                 Anish Kumar          0.642791
47   Jyotiraditya Singh Rathore          0.625223
..                          ...               ...
112                 Athak Goyal          0.256045
117    AyyappaReddy Tamalampudi          0.238418
4               Rishi Raj Singh          0.230082
125            DHANANJAY NAGPAL          0.190038
118                         NaN          0.073165

[145 rows x 2 columns]


In [61]:
# Define quintile thresholds
quantiles = ranked_resumes['similarity_score'].quantile([0.2, 0.4, 0.6, 0.8]).tolist()

def categorize_fitness(score):
    if score >= quantiles[3]:
        return 'Best Fit'
    elif score >= quantiles[2]:
        return 'Good Fit'
    elif score >= quantiles[1]:
        return 'Average Fit'
    elif score >= quantiles[0]:
        return 'Bad Fit'
    else:
        return 'Not a Fit'

ranked_resumes['fitness'] = ranked_resumes['similarity_score'].apply(categorize_fitness)

In [62]:
ranked_resumes[['Name', 'similarity_score', 'fitness']].to_csv('siamese_146_result.csv', index=True)